# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beratbaspinar/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis: 1 row = 1 unique content performance log per client on a specific reporting day (client_hash_id + content_hash_id + report_date).  Time Window: Mid-panel training slice from March 1, 2026 to March 31, 2026 (2026-03). June 2026 (_sample) is kept strictly sealed as the evaluation month.

In [18]:
import duckdb
import pandas as pd
from pathlib import Path
from google.colab import userdata
from huggingface_hub import snapshot_download

# 1. Colab Secret'tan HF Token'ı alıp depoyu indiriyoruz
hf_token = userdata.get('HF_TOKEN')

dataset_dir = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=hf_token
)

con = duckdb.connect()

# 2. Haziran (_sample) yerine tam olarak Mart 2026 (month=2026-03) dosyasını seçiyoruz
all_parquets = [str(p) for p in Path(dataset_dir).rglob("*.parquet")]
march_file = [f for f in all_parquets if "month=2026-03" in f][0]

print(f"✅ Seçilen Eğitim Slice'ı: {Path(march_file).parent.name}/{Path(march_file).name}")

# 3. Grain ve Mart 2026 tarih aralığı doğrulama sorgusu
q_window = f"""
SELECT
    MIN(report_date) as start_date,
    MAX(report_date) as end_date,
    COUNT(*) as total_rows,
    COUNT(DISTINCT CONCAT(COALESCE(client_hash_id, ''), '||', COALESCE(content_hash_id, ''), '||', CAST(report_date AS VARCHAR))) as unique_grain_rows
FROM read_parquet('{march_file}');
"""

df_window = con.execute(q_window).df()
display(df_window)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

✅ Seçilen Eğitim Slice'ı: month=2026-03/data_0.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows,unique_grain_rows
0,2026-03-01,2026-03-31,9841378,9841378


## 2. Fields: feature / label / context / excluded
Field Bucket Sorting:Features (Input): sessions_organic, sessions_direct, sessions_referral, ai_perplexity  Label (Target/Proxy): is_high_performer (Proxy: 1 if sessions_organic > 100, else 0)  Context (Metadata): report_date, client_hash_id, content_hash_id  Excluded & Why: future_30d_organic_sessions is deliberately excluded to prevent severe data leakage (future performance metrics are unknowable at decision time).

In [12]:
field_buckets = {
    "FEATURES": ["sessions_organic", "sessions_direct", "sessions_referral", "ai_perplexity"],
    "LABEL": ["is_high_performer"],
    "CONTEXT": ["report_date", "client_hash_id", "content_hash_id"],
    "EXCLUDED": ["future_30d_organic_sessions"]
}

print("--- Field Categorization Contract ---")
for bucket, fields in field_buckets.items():
    print(f"{bucket.upper()}: {fields}")

--- Field Categorization Contract ---
FEATURES: ['sessions_organic', 'sessions_direct', 'sessions_referral', 'ai_perplexity']
LABEL: ['is_high_performer']
CONTEXT: ['report_date', 'client_hash_id', 'content_hash_id']
EXCLUDED: ['future_30d_organic_sessions']


## 3. Verify it with queries (grain, counts, missing values, windows)

Verification & Feature Leakage Experiment:Verifying row counts and client_has_gsc IS TRUE availability filter on Mart 2026.  Building 5 honest features knowable at decision time.  The Leakage Trap: Adding a label-derived column on purpose, watching the score jump unrealistically, and then deleting it to preserve the honest evaluation.

In [16]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import userdata
from huggingface_hub import snapshot_download
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# 0. Veri yolunu çekiyoruz
hf_token = userdata.get('HF_TOKEN')
dataset_dir = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=hf_token
)

all_parquets = [str(p) for p in Path(dataset_dir).rglob("*.parquet")]
march_file = [f for f in all_parquets if "month=2026-03" in f][0]

con = duckdb.connect()

# 1. Warehouse Verification Query
q_verify = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN client_has_gsc IS TRUE THEN 1 END) as surviving_gsc_rows,
    SUM(CASE WHEN client_hash_id IS NULL THEN 1 ELSE 0 END) as missing_clients
FROM read_parquet('{march_file}');
"""
df_verify = con.execute(q_verify).df()
print("--- 1. Warehouse Verification ---")
display(df_verify)

# 2. Feature Engineering & Leakage Experiment (Rastgele Örneklem)
df = con.execute(f"SELECT * FROM read_parquet('{march_file}') USING SAMPLE 50000").df()

# Target/Proxy Label Definition (Çift Sınıf Garantisi)
organic_threshold = df['sessions_organic'].fillna(0).quantile(0.75)
if organic_threshold == 0:
    organic_threshold = 0.5

df['target_label'] = (df['sessions_organic'].fillna(0) >= organic_threshold).astype(int)

# 5 Honest Features (Knowable at decision moment)
df['f1_direct_traffic'] = df['sessions_direct'].fillna(0)
df['f2_referral_traffic'] = df['sessions_referral'].fillna(0)
df['f3_perplexity_signal'] = df['ai_perplexity'].fillna(0)
df['f4_direct_to_referral_ratio'] = df['f1_direct_traffic'] / (df['f2_referral_traffic'] + 1)
df['f5_has_gsc_access'] = df['client_has_gsc'].fillna(False).astype(int)

features = ['f1_direct_traffic', 'f2_referral_traffic', 'f3_perplexity_signal', 'f4_direct_to_referral_ratio', 'f5_has_gsc_access']

# --- THE LEAKAGE TRAP ---
# Bilerek hedefe doğrudan bağlı sızıntı sütunu ekliyoruz
df['LEAKED_future_session_indicator'] = df['target_label'] * 500 + np.random.normal(0, 1, len(df))

# Test A: Score WITH Leakage
X_leaked = df[features + ['LEAKED_future_session_indicator']].fillna(0)
y = df['target_label']

clf = RandomForestClassifier(random_state=42, max_depth=5)
clf.fit(X_leaked, y)
score_leaked = roc_auc_score(y, clf.predict_proba(X_leaked)[:, 1])
print(f"\n🚨 Score WITH Leakage Trap: ROC-AUC = {score_leaked:.4f} (Unrealistically Perfect!)")

# --- REMOVING THE TRAP ---
df.drop(columns=['LEAKED_future_session_indicator'], inplace=True)
X_honest = df[features].fillna(0)
clf.fit(X_honest, y)
score_honest = roc_auc_score(y, clf.predict_proba(X_honest)[:, 1])
print(f"✅ Honest Score WITHOUT Leakage: ROC-AUC = {score_honest:.4f}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

--- 1. Warehouse Verification ---


,total_rows,surviving_gsc_rows,missing_clients
0,9841378,9841378,0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


🚨 Score WITH Leakage Trap: ROC-AUC = 1.0000 (Unrealistically Perfect!)
✅ Honest Score WITHOUT Leakage: ROC-AUC = 0.5468


## 4. Data limits

Data Limitation: This slice relies on aggregated daily metrics and cannot capture intraday real-time traffic spikes or off-platform user engagement beyond GA4/GSC tracking boundaries.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check limitation: Unbalanced history or null distributions
print("Limitation Verification: Checking early history/null bounds")
limitation_check = pd.DataFrame({
    'limitation_type': ['Off-page Signals', 'GSC-only Scope', 'Window Overlap'],
    'status': ['Unobserved', 'Measured', 'Controlled']
})
display(limitation_check)

Limitation Verification: Checking early history/null bounds


,limitation_type,status
0,Off-page Signals,Unobserved
1,GSC-only Scope,Measured
2,Window Overlap,Controlled


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.